# AgroVisionAI — Testes de API, Postman e avaliação de bounding boxes

Este notebook complementa `AgroVisionAI_Modulo2_Final.ipynb` sem alterar o notebook principal.

Ele cobre três pendências:
1. Gerar/verificar `api.py` e inicializar o servidor com Uvicorn.
2. Gerar uma Collection Postman com testes automatizados e executar testes HTTP equivalentes pelo próprio notebook.
3. Avaliar quantitativamente as bounding boxes de YOLOv8, YOLOv9 e YOLOv11 usando as anotações reais do conjunto `test` (ou `valid`, se `test` não existir), além de produzir imagens comparativas.

**Importante:** os resultados só serão registrados depois da execução. Não há métricas inventadas neste arquivo. Os testes de API exigem os artefatos treinados na pasta `models/`.

## 0. Configuração do projeto

Coloque este notebook dentro de uma pasta `testes/` na raiz do projeto, ou ajuste `PROJECT_DIR` para a pasta que contém `models/`, `api.py` e o dataset de frutos.

Estrutura esperada:

```text
AgroVisionAI/
├── api.py
├── models/
│   ├── modelo_folhas.keras
│   ├── classes.json
│   ├── yolov8_best.pt
│   ├── yolov9_best.pt
│   └── yolo11_best.pt
├── dataset_frutos/
│   └── data.yaml
└── testes/
    └── testes.ipynb
```

In [ ]:
from pathlib import Path
import os, sys, json, time, subprocess, socket, textwrap, warnings
import pandas as pd
import numpy as np

# Ajuste se a pasta do projeto estiver em outro local.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "models").exists() and (PROJECT_DIR.parent / "models").exists():
    PROJECT_DIR = PROJECT_DIR.parent

MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
TESTS_DIR = PROJECT_DIR / "testes"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TESTS_DIR.mkdir(parents=True, exist_ok=True)

WEIGHTS = {
    "v8": MODELS_DIR / "yolov8_best.pt",
    "v9": MODELS_DIR / "yolov9_best.pt",
    "v11": MODELS_DIR / "yolo11_best.pt",
}
print("Projeto:", PROJECT_DIR.resolve())
print("Models:", MODELS_DIR.resolve())
print("Pesos encontrados:", {k: v.exists() for k, v in WEIGHTS.items()})

## 1. Dependências e geração do `api.py`

Instale as dependências no mesmo ambiente Python que será usado pelo servidor. O código abaixo gera `api.py` com os endpoints implementados no notebook principal, mantendo os nomes de pesos esperados.
os.chdir(PROJECT_DIR)
print('Diretório de trabalho definido para:', Path.cwd())


In [ ]:
from pathlib import Path
import json

API_CODE = '"""\nAgroVisionAI - API REST\nEndpoints:\n  POST /classificar-folha\n  POST /detectar-fruto?modelo=v8|v9|v11\n  GET  /health\nOs pesos devem estar em ./models/ e o modelo CNN em ./models/modelo_folhas.keras.\n"""\nimport json\nimport os\nfrom contextlib import asynccontextmanager\nfrom pathlib import Path\nfrom typing import Dict\n\nimport cv2\nimport numpy as np\nimport tensorflow as tf\nfrom fastapi import FastAPI, File, HTTPException, Query, UploadFile\nfrom fastapi.responses import JSONResponse\nfrom ultralytics import YOLO\n\nBASE_DIR = Path(__file__).resolve().parent\nMODELS_DIR = Path(os.getenv("AGROVISION_MODELS_DIR", BASE_DIR / "models"))\nCNN_PATH = Path(os.getenv("AGROVISION_CNN_PATH", MODELS_DIR / "modelo_folhas.keras"))\nCLASSES_PATH = Path(os.getenv("AGROVISION_CLASSES_PATH", MODELS_DIR / "classes.json"))\nMAX_IMAGE_BYTES = int(os.getenv("MAX_IMAGE_BYTES", 10 * 1024 * 1024))\n\nYOLO_FILES = {\n    "v8": Path(os.getenv("YOLO_V8_WEIGHTS", MODELS_DIR / "yolov8_best.pt")),\n    "v9": Path(os.getenv("YOLO_V9_WEIGHTS", MODELS_DIR / "yolov9_best.pt")),\n    "v11": Path(os.getenv("YOLO_V11_WEIGHTS", MODELS_DIR / "yolo11_best.pt")),\n}\n\nstate: Dict = {"cnn": None, "classes": [], "yolo": {}, "load_errors": {}}\n\n\ndef load_artifacts():\n    state["load_errors"] = {}\n    if CNN_PATH.exists() and CLASSES_PATH.exists():\n        try:\n            state["cnn"] = tf.keras.models.load_model(CNN_PATH)\n            state["classes"] = json.loads(CLASSES_PATH.read_text(encoding="utf-8"))\n        except Exception as exc:\n            state["load_errors"]["cnn"] = str(exc)\n            state["cnn"] = None\n    else:\n        state["load_errors"]["cnn"] = (\n            f"Arquivos ausentes: {CNN_PATH.name} e/ou {CLASSES_PATH.name}"\n        )\n\n    state["yolo"] = {}\n    for version, weights_path in YOLO_FILES.items():\n        if weights_path.exists():\n            try:\n                state["yolo"][version] = YOLO(str(weights_path))\n            except Exception as exc:\n                state["load_errors"][f"yolo_{version}"] = str(exc)\n        else:\n            state["load_errors"][f"yolo_{version}"] = f"Pesos não encontrados: {weights_path}"\n\n\n@asynccontextmanager\nasync def lifespan(app: FastAPI):\n    load_artifacts()\n    yield\n\n\napp = FastAPI(\n    title="AgroVisionAI",\n    description="API REST para classificação de folhas e detecção de frutos.",\n    version="2.0.0",\n    lifespan=lifespan,\n)\n\n\nasync def read_image(file: UploadFile) -> np.ndarray:\n    allowed = {"image/jpeg", "image/png", "image/jpg"}\n    if file.content_type and file.content_type.lower() not in allowed:\n        raise HTTPException(status_code=415, detail="Envie uma imagem PNG ou JPEG.")\n    content = await file.read()\n    if not content:\n        raise HTTPException(status_code=400, detail="Arquivo vazio.")\n    if len(content) > MAX_IMAGE_BYTES:\n        raise HTTPException(status_code=413, detail="Imagem excede o limite de 10 MB.")\n    image = cv2.imdecode(np.frombuffer(content, dtype=np.uint8), cv2.IMREAD_COLOR)\n    if image is None:\n        raise HTTPException(status_code=400, detail="Não foi possível decodificar a imagem.")\n    return image\n\n\n@app.get("/health")\nasync def health():\n    return {\n        "status": "healthy",\n        "cnn_loaded": state["cnn"] is not None,\n        "classes_loaded": len(state["classes"]) > 0,\n        "yolo_loaded": sorted(state["yolo"].keys()),\n        "load_errors": state["load_errors"],\n    }\n\n\n@app.post("/classificar-folha")\nasync def classificar_folha(file: UploadFile = File(...)):\n    if state["cnn"] is None:\n        raise HTTPException(\n            status_code=503,\n            detail="CNN indisponível. Treine/exporte o modelo e coloque modelo_folhas.keras e classes.json em models/.",\n        )\n    image_bgr = await read_image(file)\n    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)\n    input_shape = state["cnn"].input_shape\n    height = int(input_shape[1] or 128)\n    width = int(input_shape[2] or 128)\n    resized = cv2.resize(image_rgb, (width, height)).astype("float32") / 255.0\n    probabilities = state["cnn"].predict(np.expand_dims(resized, axis=0), verbose=0)[0]\n    class_index = int(np.argmax(probabilities))\n    if class_index >= len(state["classes"]):\n        raise HTTPException(status_code=500, detail="Mapeamento de classes incompatível com a CNN.")\n    confidence = float(probabilities[class_index])\n    class_name = state["classes"][class_index]\n    healthy = "healthy" in class_name.lower()\n    return {\n        "arquivo": file.filename,\n        "classe_id": class_index,\n        "classe": class_name,\n        "condicao": "saudavel" if healthy else "possivel_doenca",\n        "confianca": round(confidence, 6),\n        "observacao": "Predição automatizada; confirme o diagnóstico com avaliação agronômica.",\n    }\n\n\n@app.post("/detectar-fruto")\nasync def detectar_fruto(\n    file: UploadFile = File(...),\n    modelo: str = Query("v8", pattern="^(v8|v9|v11)$"),\n    confianca_minima: float = Query(0.25, ge=0.0, le=1.0),\n):\n    detector = state["yolo"].get(modelo)\n    if detector is None:\n        raise HTTPException(\n            status_code=503,\n            detail=f"Modelo {modelo} não carregado. Verifique se o peso correspondente está em {MODELS_DIR}.",\n        )\n    image_bgr = await read_image(file)\n    results = detector.predict(source=image_bgr, conf=confianca_minima, verbose=False)\n    detections = []\n    if results and results[0].boxes is not None:\n        for box in results[0].boxes:\n            x1, y1, x2, y2 = [float(v) for v in box.xyxy[0].cpu().tolist()]\n            class_id = int(box.cls[0].item())\n            detections.append({\n                "classe_id": class_id,\n                "classe": detector.names.get(class_id, str(class_id)),\n                "confianca": round(float(box.conf[0].item()), 6),\n                "bbox": {\n                    "x1": round(x1, 2), "y1": round(y1, 2),\n                    "x2": round(x2, 2), "y2": round(y2, 2),\n                },\n            })\n    return {\n        "arquivo": file.filename,\n        "modelo": modelo,\n        "total_deteccoes": len(detections),\n        "deteccoes": detections,\n    }\n\n\n@app.exception_handler(Exception)\nasync def unexpected_error_handler(request, exc):\n    # Erros HTTP previstos continuam sendo tratados pelo FastAPI.\n    if isinstance(exc, HTTPException):\n        return JSONResponse(status_code=exc.status_code, content={"detail": exc.detail})\n    return JSONResponse(status_code=500, content={"detail": "Erro interno ao processar a requisição."})\n'
Path("api.py").write_text(API_CODE, encoding="utf-8")
print("api.py final criado.")


### Verificação dos artefatos antes de iniciar

A CNN precisa de `modelo_folhas.keras` e `classes.json`. Cada detector precisa do respectivo `best.pt`. Se algum arquivo estiver ausente, a API pode iniciar, mas o endpoint correspondente responderá HTTP 503.

In [ ]:
required = {
    "CNN": MODELS_DIR / "modelo_folhas.keras",
    "classes": MODELS_DIR / "classes.json",
    **{f"YOLO {k}": v for k, v in WEIGHTS.items()}
}
status_artefatos = pd.DataFrame(
    [{"artefato": nome, "caminho": str(path), "existe": path.exists()} for nome, path in required.items()]
)
display(status_artefatos)

## 2. Inicialização real do servidor Uvicorn

A célula seguinte inicia o servidor em segundo plano, aguarda a porta 8000 e consulta `/health`. Se preferir iniciar manualmente em um terminal, use:

```bash
cd /caminho/para/AgroVisionAI
python -m uvicorn api:app --host 127.0.0.1 --port 8000
```

Não use `--reload` em ambientes de notebook/produção; para desenvolvimento local ele é opcional.

In [ ]:
# Inicialização robusta do Uvicorn — célula autocontida
import os
import sys
import time
import socket
import subprocess
import urllib.request
import urllib.error
from pathlib import Path

BASE_URL = "http://127.0.0.1:8000"
UVICORN_PROCESS = None

# Reaproveita PROJECT_DIR da célula de configuração; se não existir,
# procura a raiz do projeto a partir da pasta atual.
if "PROJECT_DIR" not in globals():
    atual = Path.cwd().resolve()
    candidatos = [atual, *atual.parents]
    PROJECT_DIR = next(
        (p for p in candidatos if (p / "api.py").exists() or (p / "models").exists()),
        atual
    )
else:
    PROJECT_DIR = Path(PROJECT_DIR).resolve()

# Se o notebook estiver sendo executado dentro da pasta testes/, sobe um nível
# quando a raiz do projeto estiver no diretório pai.
if not (PROJECT_DIR / "api.py").exists() and (PROJECT_DIR.parent / "api.py").exists():
    PROJECT_DIR = PROJECT_DIR.parent

API_FILE = PROJECT_DIR / "api.py"
if not API_FILE.exists():
    raise FileNotFoundError(
        f"Não encontrei {API_FILE}. Execute primeiro a célula que gera api.py "
        "e confirme se PROJECT_DIR aponta para a raiz do AgroVisionAI."
    )

def porta_aberta(host="127.0.0.1", port=8000):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.5)
        return sock.connect_ex((host, port)) == 0

def health_ok():
    try:
        with urllib.request.urlopen(f"{BASE_URL}/health", timeout=3) as response:
            return response.status == 200, response.read().decode("utf-8", errors="replace")
    except Exception:
        return False, ""

os.chdir(PROJECT_DIR)
print("Raiz do projeto:", PROJECT_DIR)
print("Arquivo da API:", API_FILE)

if porta_aberta():
    ok, body = health_ok()
    if ok:
        print("Já existe uma API respondendo na porta 8000. Ela será reutilizada.")
        print("Resposta /health:", body[:1000])
    else:
        raise RuntimeError(
            "A porta 8000 já está ocupada, mas o serviço não respondeu corretamente "
            "em /health. Encerre o processo que está usando a porta 8000 ou altere "
            "BASE_URL/porta antes de continuar."
        )
else:
    log_path = PROJECT_DIR / "results" / "uvicorn.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", encoding="utf-8")

    cmd = [
        sys.executable, "-m", "uvicorn", "api:app",
        "--app-dir", str(PROJECT_DIR),
        "--host", "127.0.0.1",
        "--port", "8000"
    ]
    UVICORN_PROCESS = subprocess.Popen(
        cmd,
        cwd=str(PROJECT_DIR),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True
    )

    # Aguarda a API responder, não apenas a porta abrir.
    for tentativa in range(90):
        ok, body = health_ok()
        if ok:
            print(f"Uvicorn iniciado com sucesso após {tentativa + 1} segundo(s).")
            print("Resposta /health:", body[:1000])
            break

        if UVICORN_PROCESS.poll() is not None:
            log_file.flush()
            log_file.close()
            saida = log_path.read_text(encoding="utf-8", errors="replace")
            raise RuntimeError(
                "Uvicorn encerrou antes de a API ficar disponível.\n"
                f"Consulte o log: {log_path}\n\n{saida[-7000:]}"
            )
        time.sleep(1)
    else:
        log_file.flush()
        saida = log_path.read_text(encoding="utf-8", errors="replace")
        raise TimeoutError(
            "A API não respondeu em /health após 90 segundos. "
            f"Consulte o log: {log_path}\n\n{saida[-7000:]}"
        )

print("Servidor disponível:", BASE_URL)
print("Swagger:", BASE_URL + "/docs")
print("Log do Uvicorn:", PROJECT_DIR / "results" / "uvicorn.log")


In [ ]:
import requests

health_response = requests.get(f"{BASE_URL}/health", timeout=30)
print("HTTP:", health_response.status_code)
health_json = health_response.json()
display(pd.json_normalize(health_json))
assert health_response.status_code == 200, health_json

## 3. Testes HTTP automatizados (equivalentes aos testes Postman)

Informe caminhos para uma imagem de folha do PlantVillage e uma imagem de fruto do conjunto YOLO. Os arquivos precisam existir localmente. O teste testa o endpoint de saúde, classificação, detecção para as três versões e validação de parâmetro inválido.

Os testes de imagem só passam quando os pesos e os arquivos de entrada estiverem disponíveis.

In [ ]:
# Ajuste os caminhos para imagens reais do seu projeto.
LEAF_IMAGE = None   # Ex.: PROJECT_DIR / "amostras" / "folha_teste.jpg"
FRUIT_IMAGE = None  # Ex.: PROJECT_DIR / "dataset_frutos" / "test" / "images" / "fruto.jpg"

# Descoberta automática de exemplos quando os caminhos não forem informados.
def first_image(candidates):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    for folder in candidates:
        if folder.exists():
            found = next((p for p in folder.rglob("*") if p.suffix.lower() in exts), None)
            if found:
                return found
    return None

if LEAF_IMAGE is None:
    LEAF_IMAGE = first_image([
        PROJECT_DIR / "amostras",
        PROJECT_DIR / "data" / "test",
        PROJECT_DIR / "PlantVillage"
    ])
if FRUIT_IMAGE is None:
    FRUIT_IMAGE = first_image([
        PROJECT_DIR / "dataset_frutos" / "test" / "images",
        PROJECT_DIR / "dataset_frutos" / "valid" / "images",
        PROJECT_DIR / "test" / "images",
        PROJECT_DIR / "valid" / "images"
    ])

print("Imagem folha:", LEAF_IMAGE)
print("Imagem fruto:", FRUIT_IMAGE)

In [ ]:
testes_api = []

def registrar_teste(nome, response=None, erro=None):
    testes_api.append({
        "teste": nome,
        "status_http": response.status_code if response is not None else None,
        "resultado": "PASSOU" if (response is not None and response.ok and erro is None) else "FALHOU",
        "detalhe": erro or (response.text[:500] if response is not None else "Sem resposta")
    })

# Health
try:
    r = requests.get(f"{BASE_URL}/health", timeout=30)
    registrar_teste("GET /health", r, None if r.status_code == 200 else "Esperado HTTP 200")
except Exception as e:
    registrar_teste("GET /health", erro=str(e))

# Classificação de folha
if LEAF_IMAGE and Path(LEAF_IMAGE).exists():
    try:
        with open(LEAF_IMAGE, "rb") as f:
            r = requests.post(
                f"{BASE_URL}/classificar-folha",
                files={"file": (Path(LEAF_IMAGE).name, f, "image/jpeg")},
                timeout=120
            )
        ok = r.status_code == 200 and {"classe", "confianca", "condicao"}.issubset(r.json().keys())
        registrar_teste("POST /classificar-folha", r, None if ok else "Resposta não contém os campos esperados")
    except Exception as e:
        registrar_teste("POST /classificar-folha", erro=str(e))
else:
    registrar_teste("POST /classificar-folha", erro="PENDENTE: configure LEAF_IMAGE com uma imagem real de folha.")

# Detecção para cada modelo
for versao in ("v8", "v9", "v11"):
    if FRUIT_IMAGE and Path(FRUIT_IMAGE).exists():
        try:
            with open(FRUIT_IMAGE, "rb") as f:
                r = requests.post(
                    f"{BASE_URL}/detectar-fruto",
                    params={"modelo": versao},
                    files={"file": (Path(FRUIT_IMAGE).name, f, "image/jpeg")},
                    timeout=180
                )
            ok = r.status_code == 200
            if ok:
                data = r.json()
                ok = data.get("modelo") == versao and isinstance(data.get("deteccoes"), list)
                for det in data.get("deteccoes", []):
                    ok = ok and {"classe", "classe_id", "confianca", "bbox"}.issubset(det.keys())
                    if ok:
                        ok = all(k in det["bbox"] for k in ("x1", "y1", "x2", "y2"))
            registrar_teste(f"POST /detectar-fruto?modelo={versao}", r, None if ok else "JSON/estrutura de detecções inválida")
        except Exception as e:
            registrar_teste(f"POST /detectar-fruto?modelo={versao}", erro=str(e))
    else:
        registrar_teste(f"POST /detectar-fruto?modelo={versao}", erro="PENDENTE: configure FRUIT_IMAGE com imagem real de fruto.")

# Parâmetro inválido deve ser rejeitado pela validação do FastAPI.
if FRUIT_IMAGE and Path(FRUIT_IMAGE).exists():
    try:
        with open(FRUIT_IMAGE, "rb") as f:
            r = requests.post(
                f"{BASE_URL}/detectar-fruto",
                params={"modelo": "v10"},
                files={"file": (Path(FRUIT_IMAGE).name, f, "image/jpeg")},
                timeout=60
            )
        registrar_teste("Rejeitar modelo inválido", r, None if r.status_code == 422 else "Esperado HTTP 422")
    except Exception as e:
        registrar_teste("Rejeitar modelo inválido", erro=str(e))

df_testes_api = pd.DataFrame(testes_api)
display(df_testes_api)
df_testes_api.to_csv(RESULTS_DIR / "resultados_testes_api.csv", index=False)
print("Relatório salvo em:", RESULTS_DIR / "resultados_testes_api.csv")

## 4. Collection Postman com scripts de validação

Esta célula gera uma coleção Postman v2.1 com Health Check, classificação de folha e três requisições de detecção (v8, v9 e v11). Em cada requisição há scripts que verificam o status HTTP e a estrutura do JSON.

Depois de importar o arquivo no Postman:
1. Inicie a API.
2. Abra cada requisição e selecione uma imagem no campo `file` (form-data).
3. Execute a coleção pelo Collection Runner.
4. Exporte/salve o resultado do Runner e inclua capturas de tela na entrega.

In [ ]:
def postman_request(name, method, path, test_script, with_file=False):
    req = {
        "method": method,
        "header": [],
        "url": "{{base_url}}" + path
    }
    if with_file:
        req["body"] = {
            "mode": "formdata",
            "formdata": [{"key": "file", "type": "file", "src": ""}]
        }
    return {
        "name": name,
        "request": req,
        "event": [{
            "listen": "test",
            "script": {"type": "text/javascript", "exec": test_script}
        }]
    }

collection = {
    "info": {
        "_postman_id": "b8a64f9d-52a2-4c38-9b3f-2f1b7a8d9e10",
        "name": "AgroVisionAI - Testes de API",
        "schema": "https://schema.getpostman.com/json/collection/v2.1.0/collection.json"
    },
    "variable": [{"key": "base_url", "value": "http://127.0.0.1:8000"}],
    "item": [
        postman_request("Health Check", "GET", "/health", [
            "pm.test('HTTP 200', () => pm.response.to.have.status(200));",
            "pm.test('API saudável', () => pm.expect(pm.response.json().status).to.eql('healthy'));"
        ]),
        postman_request("Classificar folha", "POST", "/classificar-folha", [
            "pm.test('HTTP 200', () => pm.response.to.have.status(200));",
            "pm.test('classe e confiança', () => { const d=pm.response.json(); pm.expect(d.classe).to.be.a('string'); pm.expect(d.confianca).to.be.a('number'); });"
        ], True)
    ] + [
        postman_request(f"Detectar frutos {v.upper()}", "POST", f"/detectar-fruto?modelo={v}", [
            "pm.test('HTTP 200', () => pm.response.to.have.status(200));",
            f"pm.test('modelo {v}', () => pm.expect(pm.response.json().modelo).to.eql('{v}'));",
            "pm.test('deteccoes é lista', () => pm.expect(pm.response.json().deteccoes).to.be.an('array'));",
            "pm.test('bbox possui coordenadas', () => pm.response.json().deteccoes.forEach(d => { pm.expect(d.bbox).to.have.all.keys('x1','y1','x2','y2'); }));"
        ], True) for v in ("v8", "v9", "v11")
    ]
}
POSTMAN_PATH = TESTS_DIR / "AgroVisionAI_Testes_Postman.json"
POSTMAN_PATH.write_text(json.dumps(collection, ensure_ascii=False, indent=2), encoding="utf-8")
print("Collection criada:", POSTMAN_PATH)

## 5. Avaliação quantitativa de bounding boxes (TP, FP, FN, IoU)

A avaliação abaixo compara as caixas previstas com as anotações YOLO reais (`class_id x_center y_center width height`, coordenadas normalizadas). O pareamento é feito por classe e IoU ≥ 0,50, com associação gulosa por maior IoU.

- **TP:** previsão pareada a uma anotação da mesma classe com IoU ≥ 0,50.
- **FP:** previsão sem anotação correspondente.
- **FN:** anotação sem previsão correspondente.
- **Precisão:** TP / (TP + FP).
- **Recall:** TP / (TP + FN).
- **F1:** média harmônica entre precisão e recall.
- **IoU médio:** média das IoUs dos pares TP.

Esta avaliação complementa, mas não substitui, mAP50 e mAP50-95 calculados pelo Ultralytics. Para comparar modelos de forma justa, use exatamente o mesmo split, imagens e limiar de confiança.

In [ ]:
from pathlib import Path
import yaml, cv2, matplotlib.pyplot as plt
from ultralytics import YOLO

# Localize data.yaml. Se não for encontrado automaticamente, informe o caminho.
DATA_YAML = None  # Ex.: PROJECT_DIR / "dataset_frutos" / "data.yaml"

def localizar_data_yaml():
    candidatos = list(PROJECT_DIR.rglob("data.yaml"))
    candidatos = [p for p in candidatos if "runs" not in str(p) and "AgroVisionAI_YOLO" not in str(p)]
    return candidatos[0] if candidatos else None

if DATA_YAML is None:
    DATA_YAML = localizar_data_yaml()
if DATA_YAML is None:
    raise FileNotFoundError("Não encontrei data.yaml. Defina DATA_YAML manualmente.")

DATA_YAML = Path(DATA_YAML).resolve()
cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
dataset_root = Path(cfg.get("path", DATA_YAML.parent))
if not dataset_root.is_absolute():
    dataset_root = (DATA_YAML.parent / dataset_root).resolve()

split_name = "test" if cfg.get("test") else ("val" if cfg.get("val") else "valid")
split_entry = cfg.get(split_name, cfg.get("valid") if split_name == "valid" else None)
if split_entry is None:
    raise ValueError("data.yaml precisa definir test ou val/valid.")
images_dir = Path(split_entry)
if not images_dir.is_absolute():
    images_dir = (dataset_root / images_dir).resolve()
labels_dir = images_dir.parent.parent / "labels" / images_dir.name
if not labels_dir.exists():
    # estrutura padrão images/test -> labels/test
    labels_dir = images_dir.parent.parent / "labels" / split_name

print("YAML:", DATA_YAML)
print("Split:", split_name)
print("Imagens:", images_dir)
print("Labels:", labels_dir)
print("Total imagens:", len([p for p in images_dir.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}]))

In [ ]:
def ler_labels_yolo(label_path, width, height):
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        cls, xc, yc, bw, bh = map(float, parts)
        x1 = (xc - bw / 2) * width
        y1 = (yc - bh / 2) * height
        x2 = (xc + bw / 2) * width
        y2 = (yc + bh / 2) * height
        boxes.append({"cls": int(cls), "xyxy": [x1, y1, x2, y2]})
    return boxes

def iou_xyxy(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def avaliar_pares(preds, gts, iou_threshold=0.50):
    candidatos = []
    for pi, pred in enumerate(preds):
        for gi, gt in enumerate(gts):
            if pred["cls"] == gt["cls"]:
                score = iou_xyxy(pred["xyxy"], gt["xyxy"])
                if score >= iou_threshold:
                    candidatos.append((score, pi, gi))
    candidatos.sort(reverse=True)
    used_p, used_g, ious = set(), set(), []
    for score, pi, gi in candidatos:
        if pi not in used_p and gi not in used_g:
            used_p.add(pi); used_g.add(gi); ious.append(score)
    tp = len(used_p)
    fp = len(preds) - tp
    fn = len(gts) - len(used_g)
    return tp, fp, fn, ious

### Executar avaliação para os três pesos

O limiar de confiança padrão é 0,25 e o IoU de pareamento é 0,50. Você pode alterar os valores para análise de sensibilidade, mas mantenha os mesmos parâmetros nas três versões durante a comparação principal.

In [ ]:
CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50
MAX_IMAGES = None  # None avalia todas as imagens; use um inteiro apenas para teste rápido.

image_paths = sorted([p for p in images_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])
if MAX_IMAGES:
    image_paths = image_paths[:MAX_IMAGES]
if not image_paths:
    raise FileNotFoundError(f"Nenhuma imagem encontrada em {images_dir}")

resultados_bbox = []
erros_por_modelo = {}
for versao, peso in WEIGHTS.items():
    if not peso.exists():
        print(f"Pulando {versao}: peso ausente em {peso}")
        continue
    detector = YOLO(str(peso))
    tp_total = fp_total = fn_total = 0
    ious_total = []
    erros_por_modelo[versao] = []

    for img_path in image_paths:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        label_path = labels_dir / f"{img_path.stem}.txt"
        gts = ler_labels_yolo(label_path, w, h)

        pred_result = detector.predict(
            source=str(img_path), conf=CONF_THRESHOLD, iou=0.7, verbose=False
        )[0]
        preds = []
        if pred_result.boxes is not None:
            for box in pred_result.boxes:
                preds.append({
                    "cls": int(box.cls.item()),
                    "xyxy": [float(x) for x in box.xyxy[0].cpu().tolist()],
                    "conf": float(box.conf.item())
                })

        tp, fp, fn, ious = avaliar_pares(preds, gts, IOU_THRESHOLD)
        tp_total += tp; fp_total += fp; fn_total += fn; ious_total.extend(ious)
        if fp or fn:
            erros_por_modelo[versao].append({
                "imagem": img_path.name, "TP": tp, "FP": fp, "FN": fn
            })

    precision = tp_total / (tp_total + fp_total) if tp_total + fp_total else 0.0
    recall = tp_total / (tp_total + fn_total) if tp_total + fn_total else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    resultados_bbox.append({
        "modelo": versao, "imagens_avaliadas": len(image_paths),
        "TP": tp_total, "FP": fp_total, "FN": fn_total,
        "precisao_iou50": precision, "recall_iou50": recall,
        "f1_iou50": f1,
        "iou_medio_TP": float(np.mean(ious_total)) if ious_total else 0.0,
        "limiar_confianca": CONF_THRESHOLD, "limiar_iou": IOU_THRESHOLD
    })

df_bbox = pd.DataFrame(resultados_bbox)
display(df_bbox)
df_bbox.to_csv(RESULTS_DIR / "avaliacao_bounding_boxes_yolo.csv", index=False)
with open(RESULTS_DIR / "erros_por_imagem_yolo.json", "w", encoding="utf-8") as f:
    json.dump(erros_por_modelo, f, ensure_ascii=False, indent=2)
print("Relatórios salvos em results/.")

### Visualização comparativa: ground truth e previsões

A célula abaixo mostra uma imagem de exemplo, as anotações reais e as caixas previstas de cada modelo. A visualização facilita identificar caixas deslocadas, falsos positivos e falsos negativos.

In [ ]:
def desenhar_gt_pred(img_path, versao, conf=CONF_THRESHOLD):
    img = cv2.imread(str(img_path))
    if img is None:
        raise ValueError(f"Não foi possível abrir {img_path}")
    h, w = img.shape[:2]
    gt_path = labels_dir / f"{Path(img_path).stem}.txt"
    gts = ler_labels_yolo(gt_path, w, h)
    detector = YOLO(str(WEIGHTS[versao]))
    pred = detector.predict(source=str(img_path), conf=conf, verbose=False)[0]

    canvas = img.copy()
    # Ground truth em verde
    for gt in gts:
        x1,y1,x2,y2 = map(int, gt["xyxy"])
        cv2.rectangle(canvas, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(canvas, f"GT:{gt['cls']}", (x1,max(15,y1-5)), cv2.FONT_HERSHEY_SIMPLEX, .5, (0,255,0), 1)
    # Previsões em vermelho
    if pred.boxes is not None:
        for box in pred.boxes:
            x1,y1,x2,y2 = map(int, box.xyxy[0].cpu().tolist())
            cls = int(box.cls.item()); score = float(box.conf.item())
            cv2.rectangle(canvas, (x1,y1), (x2,y2), (0,0,255), 2)
            cv2.putText(canvas, f"Pred:{cls} {score:.2f}", (x1,min(h-5,y2+15)), cv2.FONT_HERSHEY_SIMPLEX, .5, (0,0,255), 1)
    return cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)

imagem_exemplo = image_paths[0]
for versao in WEIGHTS:
    if WEIGHTS[versao].exists():
        plt.figure(figsize=(12,8))
        plt.imshow(desenhar_gt_pred(imagem_exemplo, versao))
        plt.title(f"Ground truth (verde) x {versao.upper()} (vermelho) — {imagem_exemplo.name}")
        plt.axis("off")
        plt.show()

## 6. Comparação com métricas oficiais do Ultralytics (mAP)

Além da avaliação TP/FP/FN acima, execute `model.val()` para obter mAP50 e mAP50-95 no mesmo split. O resultado oficial depende das configurações do dataset e do Ultralytics; registre o split usado.

In [ ]:
from ultralytics import YOLO

metricas_map = []
for versao, peso in WEIGHTS.items():
    if not peso.exists():
        continue
    model = YOLO(str(peso))
    metrics = model.val(
        data=str(DATA_YAML),
        split=split_name if split_name in cfg else "val",
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        plots=True,
        project=str(RESULTS_DIR / "validacao_ultralytics"),
        name=versao,
        exist_ok=True
    )
    metricas_map.append({
        "modelo": versao,
        "precision_ultralytics": float(metrics.box.mp),
        "recall_ultralytics": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50_95": float(metrics.box.map)
    })

df_map = pd.DataFrame(metricas_map)
display(df_map)
df_map.to_csv(RESULTS_DIR / "metricas_map_ultralytics.csv", index=False)

## 7. Relatório final automático

O relatório reúne resultados de API e bounding boxes. Uma execução incompleta deve permanecer identificada como pendente/falha; não substitua valores ausentes por métricas estimadas.

In [ ]:
relatorio = {
    "projeto": "AgroVisionAI",
    "data_execucao": pd.Timestamp.now().isoformat(),
    "api": testes_api,
    "bounding_boxes": resultados_bbox if "resultados_bbox" in globals() else [],
    "metricas_ultralytics": metricas_map if "metricas_map" in globals() else [],
    "observacao": "Resultados gerados pela execução deste notebook; conferir split, pesos e imagens usados."
}
RELATORIO_PATH = RESULTS_DIR / "relatorio_testes_complementares.json"
RELATORIO_PATH.write_text(json.dumps(relatorio, ensure_ascii=False, indent=2), encoding="utf-8")
print("Relatório final:", RELATORIO_PATH)

## 8. Checklist de evidências para a submissão

- [ ] `uvicorn api:app --host 127.0.0.1 --port 8000` inicia sem erro.
- [ ] `/docs` abre no navegador.
- [ ] `/health` retorna HTTP 200 e informa os modelos carregados.
- [ ] `/classificar-folha` retorna classe, condição e confiança.
- [ ] `/detectar-fruto?modelo=v8`, `v9` e `v11` retornam JSON com coordenadas.
- [ ] Collection `AgroVisionAI_Testes_Postman.json` executada no Postman.
- [ ] Capturas de tela do Postman/Swagger salvas em `testes/evidencias/`.
- [ ] `avaliacao_bounding_boxes_yolo.csv` e imagens comparativas geradas.
- [ ] `metricas_map_ultralytics.csv` gerado no mesmo split para os três modelos.

**Interpretação:** se houver HTTP 503, verifique os caminhos/nomes dos pesos e da CNN. Se o resultado de bounding boxes indicar muitos FN, investigue limiar de confiança, qualidade das anotações, classes e domínio das imagens. Se houver muitos FP, avalie aumentar o limiar de confiança e comparar precisão/recall, sem alterar o limiar apenas para melhorar artificialmente o resultado.